# Mini-batch gradient descent & epochs

_Deep Learning_

**Update on small chunks of data instead of one example or the whole set.**

You can update the weights using one example, all examples, or a small chunk.

     A mini-batch is a small chunk (like 32 or 128 examples). It is the sweet spot.

---

This notebook builds mini-batch training up one step at a time. Run each cell, read the note above it, and watch how chunking the data into small batches changes the training loop. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We'll train a tiny linear model with mini-batch gradient descent. We build it in three steps: (1) wrap the data in a `DataLoader` that hands out small batches, (2) set up the model and optimizer, (3) loop over epochs and batches.

### Step 1 — Wrap the data so it streams in small batches

We make 1000 random `(x, y)` pairs and hand them to a `DataLoader`. The loader's job is to slice the dataset into **mini-batches** of 100 examples and reshuffle them each epoch. With 1000 examples and a batch size of 100 that gives 10 batches — so one full pass (one epoch) means 10 weight updates, not one.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(1000, 4)              # N = 1000 examples, 4 features each
y = torch.randn(1000, 1)             # one regression target per example

dataset = TensorDataset(X, y)        # pairs each X row with its y
loader = DataLoader(dataset, batch_size=100, shuffle=True)   # 1000/100 = 10 batches per epoch

### Step 2 — Set up the model, optimizer, and loss

The model is a single linear layer mapping 4 inputs to 1 output. `Adam` is the optimizer that applies each gradient update, with a learning rate of `0.01`. `MSELoss` measures how far the predictions are from the targets — the quantity we want to shrink.

In [ ]:
model = nn.Linear(4, 1)                          # 4 inputs -> 1 output
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()                           # mean-squared-error loss

### Step 3 — Train over epochs, one mini-batch at a time

An **epoch** is one full sweep through all the data. Inside each epoch we iterate over the loader, which yields one mini-batch `(xb, yb)` at a time. For every batch we zero out old gradients, compute the loss on just that batch, backpropagate, and take an optimizer step. So each epoch performs 10 updates instead of waiting for the whole dataset.

In [ ]:
for epoch in range(3):                # 3 epochs = 3 full passes over the data
    for xb, yb in loader:             # one mini-batch per iteration
        opt.zero_grad()               # clear gradients left over from last step
        predictions = model(xb)       # forward pass on just this batch
        loss = loss_fn(predictions, yb)   # loss on just this batch
        loss.backward()               # backprop: compute gradients
        opt.step()                    # nudge the weights downhill
    print("epoch", epoch, "last-batch loss", round(loss.item(), 4))

## Visualize the data & results

_On real digits, does training on small mini-batches drop the loss faster than the full batch?_

### Step 1 — Load the real digits and scale the pixels

We load the 1797 handwritten digits and scale every pixel from its 0–16 range down to 0–1. Keeping features in a small, consistent range helps gradient descent take well-behaved steps.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X = digits.data / 16.0      # pixel intensities scaled to 0..1
y = digits.target           # the digit label 0..9

### Step 2 — Train two networks: full-batch vs mini-batch

We fit the *same* small network twice. The first uses the **whole** dataset as one batch per update (`batch_size=len(X)`), so it takes a single, carefully-averaged step per epoch. The second uses **mini-batches of 32**, taking many noisier steps per epoch. We give the full-batch run a larger learning rate since it updates far less often.

In [ ]:
full = MLPClassifier(hidden_layer_sizes=(32,), solver="sgd", max_iter=40,
                     random_state=0, batch_size=len(X),
                     learning_rate_init=0.5, momentum=0.0).fit(X, y)

mini = MLPClassifier(hidden_layer_sizes=(32,), solver="sgd", max_iter=40,
                     random_state=0, batch_size=32,
                     learning_rate_init=0.05, momentum=0.0).fit(X, y)

### Step 3 — Plot both learning curves

Each network records its training loss per epoch in `loss_curve_`. Plotting them together shows the trade-off: the mini-batch run usually drives the loss down faster because it makes many updates per epoch, while the full-batch run takes one bigger step at a time.

In [ ]:
plt.plot(full.loss_curve_, color="#4ea1ff", label="full-batch (1437 samples)")
plt.plot(mini.loss_curve_, color="#ffb454", label="mini-batch (32 samples)")
plt.title("Real full-batch vs mini-batch loss on load_digits")
plt.xlabel("epoch")
plt.ylabel("log loss")
plt.legend()
plt.show()